# Time-Series Anomaly Detection Using LSTM Autoencoder Neural Network
**Course:** BSc CSIT 6th Semester — Neural Networks Project  
**Primary Objective:** Learn the normal temporal behavior of a time-series dataset using an LSTM Autoencoder and detect anomalous sequences via reconstruction error.

### Complete Pipeline Architecture:
```
Raw Time-Series Data
  -> Data Cleaning & Chronological Sorting
  -> Min-Max Normalization (Fit on Train Only)
  -> Sliding-Window Sequence Generation (Configurable window_size)
  -> Chronological Splitting (Train 60%, Val 20%, Test 20%)
  -> LSTM Autoencoder Model (Encoder -> Latent Bottleneck -> Decoder)
  -> Training on Normal Dynamics (MSE Loss + Backpropagation + Adam)
  -> Reconstruction of Test Sequences
  -> Reconstruction Error Computation (MSE per sequence)
  -> Principled Threshold Selection (Percentile / Gaussian / IQR)
  -> Anomaly Detection & Evaluation (Precision, Recall, F1, Confusion Matrix)
  -> Visualization & Interpretation
```

In [ ]:
import sys
sys.path.append('..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from utils.utils import set_seed, get_device, load_config
from preprocessing.preprocess import preprocess_pipeline, detect_columns, clean_data
from models.lstm_autoencoder import LSTMAutoencoder
from training.train import train_model, load_checkpoint
from evaluation.evaluate import (
    compute_reconstruction_errors,
    reconstruct_sequences,
    calculate_threshold,
    detect_anomalies,
    calculate_metrics,
    build_results_dataframe
)
from visualization.plots import (
    plot_raw_time_series_matplotlib,
    plot_loss_curves_matplotlib,
    plot_reconstruction_matplotlib,
    plot_reconstruction_error_matplotlib,
    plot_detected_anomalies_matplotlib,
    plot_confusion_matrix_matplotlib
)

set_seed(42)
device = get_device()
print(f'Active Device: {device}')

## 1. Exploratory Data Analysis (EDA)
We use the **Numenta Anomaly Benchmark (NAB) NYC Taxi** dataset.
The series records passenger counts aggregated in 30-minute bins from July 2014 to January 2015.

In [ ]:
df_raw = pd.read_csv('../data/raw/nyc_taxi.csv')
print(f'Dataset Shape: {df_raw.shape}')
print(df_raw.head())
print('\nSummary Statistics:\n', df_raw.describe())

with open('../data/raw/nyc_taxi_labels.json') as f:
    labels_info = json.load(f)
print(f'Known Anomaly Windows: {len(labels_info["anomaly_windows"])}')

## 2. Preprocessing & Chronological Splitting
**Scientific Principle:** We strictly avoid random shuffling of time-series observations.
Random shuffling causes *temporal data leakage* because future points would leak into the training set.
We use:
- **Earlier 60%** -> Training Set
- **Middle 20%** -> Validation Set
- **Later 20%** -> Test Set

The `MinMaxScaler` is fit *only* on the training split to prevent scale leakage.

In [ ]:
window_size = 24  # 12 hours (24 * 30-min bins)
prep_data = preprocess_pipeline(
    df_raw=df_raw,
    window_size=window_size,
    step_size=1,
    train_ratio=0.60,
    val_ratio=0.20,
    test_ratio=0.20,
    filter_anomalies_from_train=True,
    anomaly_windows=labels_info.get('anomaly_windows', []),
    anomaly_timestamps=labels_info.get('anomaly_timestamps', []),
)
print(f'X_train Shape: {prep_data.X_train.shape}')
print(f'X_val Shape:   {prep_data.X_val.shape}')
print(f'X_test Shape:  {prep_data.X_test.shape}')

## 3. LSTM Autoencoder Neural Network
The network architecture consists of an **LSTM Encoder**, a **Latent Representation Bottleneck**, and an **LSTM Decoder**.
- **Encoder:** Compresses the $(T=24, D=1)$ sequence into a latent code $z \in \mathbb{R}^{16}$.
- **Decoder:** Expands the compressed latent vector back to the original sequence length $(T=24, D=1)$ using sequential recurrence and linear projection.

In [ ]:
model = LSTMAutoencoder(
    input_dim=1,
    hidden_dims=[64, 32],
    latent_dim=16,
    seq_len=window_size,
    dropout=0.1
)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable Parameters: {total_params:,d}')

## 4. Model Training & Loss Convergence
We train the model using **Mean Squared Error (MSE)** loss and the **Adam** optimizer.
Early stopping is configured with patience=10 to prevent overfitting.

In [ ]:
trained_model, history = train_model(
    model=model,
    X_train=prep_data.X_train,
    X_val=prep_data.X_val,
    epochs=25,
    batch_size=64,
    learning_rate=0.001,
    patience=8,
    device=device,
    verbose=True
)

# Plot Loss Curve
fig_loss = plot_loss_curves_matplotlib(history)
plt.show()

## 5. Reconstruction Error & Principled Thresholding
The core premise: the model learns normal temporal patterns.
When given normal data, $\text{MSE}(X, \hat{X})$ is small.
When given anomalous sequences (e.g. Blizzard of Jan 2015), the autoencoder fails to reconstruct the unusual pattern, producing high MSE.

We compute the threshold from the **Validation set** (e.g. 95th percentile) to avoid leaking test labels into threshold selection.

In [ ]:
val_errors = compute_reconstruction_errors(trained_model, prep_data.X_val, device=device)
test_errors = compute_reconstruction_errors(trained_model, prep_data.X_test, device=device)
test_recon = reconstruct_sequences(trained_model, prep_data.X_test, device=device)

threshold = calculate_threshold(val_errors, method='percentile', percentile=95.0, val_labels=prep_data.val_labels)
print(f'Calculated Anomaly Decision Threshold (Normal Validation Data): {threshold:.6f}')

## 6. Anomaly Detection & Evaluation on Unseen Test Split
We apply the threshold to the test set and evaluate against ground truth labels.
Because anomaly detection is highly imbalanced, we focus on **Precision, Recall, and F1-Score** rather than Accuracy alone.

In [ ]:
test_preds = detect_anomalies(test_errors, threshold)
metrics = calculate_metrics(prep_data.test_labels, test_preds)

print('Test Set Performance:')
print(f'  Accuracy:  {metrics["accuracy"]:.4f}')
print(f'  Precision: {metrics["precision"]:.4f}')
print(f'  Recall:    {metrics["recall"]:.4f}')
print(f'  F1 Score:  {metrics["f1"]:.4f}')
print(f'  Confusion Matrix: TP={metrics["tp"]}, FP={metrics["fp"]}, TN={metrics["tn"]}, FN={metrics["fn"]}')

fig_cm = plot_confusion_matrix_matplotlib(metrics['confusion_matrix'])
plt.show()

## 7. Visualizing Reconstructions & Detected Anomalies

In [ ]:
# Reconstruction overlay
actual_unscaled = prep_data.scaler.inverse_transform(prep_data.X_test[:, -1, 0].reshape(-1, 1)).flatten()
recon_unscaled = prep_data.scaler.inverse_transform(test_recon[:, -1, 0].reshape(-1, 1)).flatten()

fig_overlay = plot_reconstruction_matplotlib(
    prep_data.test_timestamps,
    actual_unscaled,
    recon_unscaled,
    num_points=300,
    title='Original vs LSTM Reconstructed Signal'
)
plt.show()

# Reconstruction Error and Threshold
fig_err = plot_reconstruction_error_matplotlib(
    prep_data.test_timestamps,
    test_errors,
    threshold
)
plt.show()

# Detected Anomalies on Test Set
fig_anom = plot_detected_anomalies_matplotlib(
    prep_data.test_timestamps,
    actual_unscaled,
    test_preds,
    ground_truth=prep_data.test_labels
)
plt.show()

## 8. Summary & Viva Q&A Guide
### Core Viva Questions:
1. **Why use an LSTM instead of a standard Autoencoder?**
   *Standard feedforward networks ignore temporal autocorrelation. LSTMs maintain an internal cell state $c_t$ and hidden state $h_t$ that explicitly model temporal transitions across the sliding window.*
2. **Why does reconstruction error indicate anomalies?**
   *Because the autoencoder is trained exclusively on normal patterns. Its weights capture the normal manifold. Out-of-distribution patterns cannot be compressed and reconstructed accurately, causing high MSE.*
3. **Why use chronological splitting instead of random train/test split?**
   *Random shuffling causes future points to leak into training, artificially inflating metrics. Chronological splitting mirrors real-world deployment.*